# Demo de inferencia del modelo MLOps — PlantVillage

Este notebook carga el modelo previamente entrenado y empaquetado mediante **Joblib**, reconstruye el pipeline de **Scikit-learn** y permite realizar una predicción sobre una nueva imagen de una hoja de tomate.

El flujo de inferencia es:

$$
\text{Imagen}
\rightarrow
\text{Extracción de características}
\rightarrow
\text{Pipeline}
\rightarrow
\text{Random Forest}
\rightarrow
P(Y=c_k\mid X)
\rightarrow
\hat{y}
$$

No se realiza un nuevo entrenamiento. Se utiliza directamente el artefacto `model.joblib` generado durante la Fase 2 del proyecto.

## Archivos requeridos

Antes de ejecutar la demo debe disponer de:

- `model.joblib`
- `train_metrics.json`
- `transformers.py`
- `prueba_modelo_plantvillage.py`

La estructura final esperada en Colab será:

```text
/content/
├── models/
│   ├── model.joblib
│   └── train_metrics.json
├── src/
│   ├── __init__.py
│   └── transformers.py
└── prueba_modelo_plantvillage.py
```


# 1. Instalación de dependencias

Se instalan las bibliotecas necesarias para cargar el pipeline, procesar la imagen y ejecutar la predicción. Se fijan las versiones de `scikit-learn` y `joblib` utilizadas durante el desarrollo para reducir riesgos de incompatibilidad al deserializar el modelo.

In [ ]:
# ============================================================
# 1. INSTALACIÓN DE DEPENDENCIAS
# ============================================================

!pip install -q \
    scikit-learn==1.6.1 \
    joblib==1.5.3 \
    pandas \
    numpy \
    pillow \
    opencv-python-headless

print("=" * 80)
print("DEPENDENCIAS INSTALADAS")
print("=" * 80)


# 2. Verificación del entorno

Se comprueba que las versiones principales estén disponibles antes de cargar los artefactos del modelo.

In [ ]:
# ============================================================
# 2. VERIFICACIÓN DEL ENTORNO
# ============================================================

import sys
import numpy as np
import pandas as pd
import sklearn
import joblib
import cv2
import PIL

print("=" * 80)
print("VERIFICACIÓN DEL ENTORNO")
print("=" * 80)

print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")
print(f"Joblib       : {joblib.__version__}")
print(f"OpenCV       : {cv2.__version__}")
print(f"Pillow       : {PIL.__version__}")

print("=" * 80)


# 3. Crear la estructura del proyecto

Se crean las carpetas `models/` y `src/`, además de `src/__init__.py`, para que Python pueda importar correctamente el transformer personalizado utilizado por el pipeline.

In [ ]:
# ============================================================
# 3. CREACIÓN DE ESTRUCTURA DEL PROYECTO
# ============================================================

from pathlib import Path

BASE = Path("/content")
MODELS_DIR = BASE / "models"
SRC_DIR = BASE / "src"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

INIT_PATH = SRC_DIR / "__init__.py"
INIT_PATH.touch(exist_ok=True)

print("=" * 80)
print("ESTRUCTURA CREADA")
print("=" * 80)
print("/content/")
print("├── models/")
print("├── src/")
print("└── prueba_modelo_plantvillage.py")


# 4. Subir los archivos del proyecto

Seleccione simultáneamente los cuatro archivos requeridos:

1. `model.joblib`
2. `train_metrics.json`
3. `transformers.py`
4. `prueba_modelo_plantvillage.py`


In [ ]:
# ============================================================
# 4. CARGA DE ARCHIVOS DEL PROYECTO
# ============================================================

from google.colab import files

print("Seleccione estos cuatro archivos:")
print("1. model.joblib")
print("2. train_metrics.json")
print("3. transformers.py")
print("4. prueba_modelo_plantvillage.py")
print()

archivos_subidos = files.upload()

print("\nArchivos recibidos:")
for nombre in archivos_subidos:
    print(f" - {nombre}")


# 5. Organizar los archivos

Los archivos cargados por Colab se ubican inicialmente en `/content`. Esta celda los mueve a la estructura esperada por el script.

In [ ]:
# ============================================================
# 5. ORGANIZACIÓN DE LOS ARCHIVOS
# ============================================================

from pathlib import Path
import shutil

BASE = Path("/content")

destinos = {
    "model.joblib": BASE / "models" / "model.joblib",
    "train_metrics.json": BASE / "models" / "train_metrics.json",
    "transformers.py": BASE / "src" / "transformers.py",
    "prueba_modelo_plantvillage.py": BASE / "prueba_modelo_plantvillage.py",
}

for nombre, destino in destinos.items():
    origen = BASE / nombre

    if origen.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)

        if origen.resolve() != destino.resolve():
            if destino.exists():
                destino.unlink()
            shutil.move(str(origen), str(destino))

        print(f"[OK] {nombre} -> {destino}")

    elif destino.exists():
        print(f"[OK] {nombre} ya estaba en {destino}")

    else:
        print(f"[ERROR] No se encontró {nombre}")


# 6. Verificación de archivos

Antes de cargar el modelo, se comprueba que todos los artefactos necesarios existan en sus rutas definitivas.

In [ ]:
# ============================================================
# 6. VERIFICACIÓN DE LA ESTRUCTURA
# ============================================================

from pathlib import Path

archivos_requeridos = [
    Path("/content/models/model.joblib"),
    Path("/content/models/train_metrics.json"),
    Path("/content/src/transformers.py"),
    Path("/content/src/__init__.py"),
    Path("/content/prueba_modelo_plantvillage.py"),
]

print("=" * 90)
print("VERIFICACIÓN DE ARCHIVOS")
print("=" * 90)

todo_correcto = True

for archivo in archivos_requeridos:
    existe = archivo.exists()
    estado = "[OK]" if existe else "[FALTA]"
    print(f"{estado:<8} {archivo}")
    if not existe:
        todo_correcto = False

print("=" * 90)

if todo_correcto:
    print("Todos los archivos necesarios están disponibles.")
else:
    raise FileNotFoundError(
        "Faltan archivos. No continúe hasta corregir la estructura."
    )


# 7. Inspección de metadatos del modelo

Se revisa el archivo `train_metrics.json` para comprobar el modelo seleccionado, la versión registrada, el número de features y el número de clases.

In [ ]:
# ============================================================
# 7. INSPECCIÓN DE LOS METADATOS DEL MODELO
# ============================================================

import json

METADATA_PATH = "/content/models/train_metrics.json"

with open(METADATA_PATH, "r", encoding="utf-8") as archivo:
    metadata = json.load(archivo)

modelo_seleccionado = (
    metadata.get("model", {}).get("selected_model", "No informado")
)

version_modelo = (
    metadata.get("artifact", {}).get("model_version", "No informada")
)

features_input = (
    metadata.get("features", {}).get("input", [])
)

clases = (
    metadata.get("target", {}).get("classes", [])
)

print("=" * 90)
print("METADATOS DEL MODELO")
print("=" * 90)
print(f"Modelo seleccionado : {modelo_seleccionado}")
print(f"Versión             : {version_modelo}")
print(f"Número de features  : {len(features_input)}")
print(f"Número de clases    : {len(clases)}")
print("=" * 90)

if features_input:
    print("\nFeatures registradas:")
    for i, feature in enumerate(features_input, start=1):
        print(f"{i:02d}. {feature}")


# 8. Verificación del transformer personalizado

El modelo serializado contiene una clase personalizada llamada `FeatureEngineeringProduccion`. Por este motivo, `src/transformers.py` debe poder importarse antes de ejecutar `joblib.load()`.

In [ ]:
# ============================================================
# 8. VERIFICACIÓN DEL TRANSFORMER PERSONALIZADO
# ============================================================

import sys

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from src.transformers import FeatureEngineeringProduccion

print("=" * 90)
print("TRANSFORMER PERSONALIZADO")
print("=" * 90)
print("Clase importada correctamente:")
print(FeatureEngineeringProduccion)
print("=" * 90)


# 9. Control previo a la demo

Esta comprobación final permite verificar que el entorno está listo antes de ejecutar la prueba interactiva.

In [ ]:
# ============================================================
# 9. CONTROL PREVIO A LA DEMO
# ============================================================

from pathlib import Path

elementos = {
    "Modelo": Path("/content/models/model.joblib"),
    "Metadatos": Path("/content/models/train_metrics.json"),
    "Transformer": Path("/content/src/transformers.py"),
    "Script": Path("/content/prueba_modelo_plantvillage.py"),
}

print("=" * 80)
print("CONTROL PREVIO A LA DEMO")
print("=" * 80)

todo_ok = True

for nombre, ruta in elementos.items():
    estado = "OK" if ruta.exists() else "FALTA"
    print(f"{nombre:<15}: {estado:<6} {ruta}")
    if not ruta.exists():
        todo_ok = False

print("=" * 80)

if not todo_ok:
    raise FileNotFoundError(
        "La demo no puede ejecutarse porque faltan artefactos."
    )

print("Entorno listo para ejecutar la inferencia.")


# 10. Ejecutar la demo interactiva

Esta celda ejecuta `prueba_modelo_plantvillage.py`.

El script:

1. localiza el proyecto;
2. carga `train_metrics.json`;
3. verifica el contrato de 15 features;
4. verifica el SHA-256 del modelo, cuando está disponible;
5. carga `model.joblib`;
6. solicita una imagen de prueba;
7. extrae las 15 características visuales;
8. ejecuta `predict()` y `predict_proba()`;
9. muestra clase predicha, confianza y Top-3 de probabilidades;
10. permite comparar opcionalmente la predicción con la clase real.

El flujo es:

$$
\text{Imagen}
\rightarrow
\mathbf{x}\in\mathbb{R}^{15}
\rightarrow
\text{Pipeline}
\rightarrow
\text{Random Forest}
\rightarrow
\hat{y}
$$


In [ ]:
# ============================================================
# 10. EJECUCIÓN DE LA DEMO
# ============================================================

%run /content/prueba_modelo_plantvillage.py


# 11. Repetir una nueva prueba

Para clasificar otra imagen, vuelva a ejecutar únicamente la celda anterior:

```python
%run /content/prueba_modelo_plantvillage.py
```

No es necesario reinstalar librerías, volver a cargar los artefactos ni reentrenar el modelo.


# 12. Interpretación de la salida

Para una imagen nueva, el modelo produce un vector de probabilidades:

$$
\mathbf{p} =
[p_1,p_2,\ldots,p_{10}]
$$

donde:

$$
p_k = P(Y=c_k\mid X)
$$

y se verifica:

$$
\sum_{k=1}^{10}p_k \approx 1.
$$

La clase final corresponde a:

$$
\hat{y}
=
\arg\max_k p_k.
$$

La **confianza** mostrada por la demo corresponde a la mayor probabilidad estimada.

Durante la inferencia, la clase real no forma parte de las variables de entrada:

$$
Y_{\text{real}}\notin X_{\text{inferencia}}.
$$

Por tanto, la demostración respeta la prevención de fuga de información utilizada durante el desarrollo del proyecto.
